# Universal Drive Uploader - Web Server

### Flask API server for web UI integration

**Features:**
- REST API with SSE progress streaming
- Multi-threaded download queue
- Direct downloads + yt-dlp video support
- Google Drive folder management
- Pinggy.io tunnel for remote access

**Setup:**
1. Run cells 1-3 to initialize
2. Cell 4 starts the Flask server
3. Cell 5 starts Pinggy tunnel (copy the URL)
4. Use the tunnel URL in your web UI

---

In [ ]:
#@title 1. Setup & Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
%pip install -q flask flask-cors yt-dlp requests tqdm

print("Setup complete!")

In [ ]:
#@title 2. Configuration & Utilities

import os
import re
import time
import mimetypes
from urllib.parse import urlparse, unquote
from datetime import timedelta

import requests
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError


# ==================== CONFIGURATION ====================
class Config:
    """Server configuration."""
    DRIVE_BASE = "/content/drive/MyDrive"
    DEFAULT_FOLDER = "Downloads"
    CHUNK_SIZE = 50 * 1024 * 1024  # 50MB
    MAX_RETRIES = 5
    RETRY_DELAY = 3
    TIMEOUT = 60
    USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36")
    SCOPES = ['https://www.googleapis.com/auth/drive']
    TOKEN_PATH = '/content/drive/MyDrive/token.json'
    CREDENTIALS_PATH = '/content/drive/MyDrive/credentials.json'


# ==================== UTILITY FUNCTIONS ====================
def format_size(size_bytes):
    """Convert bytes to human-readable format.
    
    Args:
        size_bytes: Size in bytes
        
    Returns:
        str: Formatted size string (e.g., '1.5 GB')
    """
    if size_bytes is None or size_bytes == 0:
        return "0 B"
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} PB"


def format_speed(bytes_per_sec):
    """Convert bytes/sec to human-readable format.
    
    Args:
        bytes_per_sec: Speed in bytes per second
        
    Returns:
        str: Formatted speed string (e.g., '10.5 MB/s')
    """
    if bytes_per_sec is None or bytes_per_sec == 0:
        return "0 B/s"
    return format_size(bytes_per_sec) + "/s"


def sanitize_filename(filename):
    """Remove invalid characters from filename.
    
    Args:
        filename: Original filename
        
    Returns:
        str: Sanitized filename
    """
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    filename = filename.strip(' .')
    if len(filename) > 200:
        name, ext = os.path.splitext(filename)
        filename = name[:200-len(ext)] + ext
    return filename or "downloaded_file"


def get_filename_from_url(url, response=None):
    """Extract filename from URL or response headers.
    
    Args:
        url: Source URL
        response: Optional requests.Response object
        
    Returns:
        str: Extracted filename or None
    """
    if response and 'Content-Disposition' in response.headers:
        cd = response.headers['Content-Disposition']
        matches = re.findall(
            r'filename[*]?=["\']?(?:UTF-8\'\')?([^"\';<>]+)',
            cd, re.IGNORECASE
        )
        if matches:
            return sanitize_filename(unquote(matches[0]))

    parsed = urlparse(url)
    path = unquote(parsed.path)
    filename = os.path.basename(path)

    if filename and '.' in filename:
        return sanitize_filename(filename)

    return None


def detect_url_type(url):
    """Detect the type of URL for appropriate handling.
    
    Args:
        url: URL to analyze
        
    Returns:
        str: URL type ('video', 'torrent', 'mega', 'gdrive', or 'direct')
    """
    url_lower = url.lower()

    if url_lower.startswith('magnet:'):
        return 'torrent'
    if url_lower.endswith('.torrent'):
        return 'torrent'
    if 'mega.nz' in url_lower or 'mega.co.nz' in url_lower:
        return 'mega'
    if 'drive.google.com' in url_lower:
        return 'gdrive'

    video_domains = [
        'youtube.com', 'youtu.be', 'twitter.com', 'x.com', 'instagram.com',
        'tiktok.com', 'reddit.com', 'twitch.tv', 'vimeo.com', 'dailymotion.com',
        'facebook.com', 'fb.watch', 'soundcloud.com', 'bandcamp.com',
        'bilibili.com', 'nicovideo.jp'
    ]
    for domain in video_domains:
        if domain in url_lower:
            return 'video'

    return 'direct'


# ==================== DRIVE API ====================
def get_drive_service():
    """Get authenticated Drive API service.
    
    Returns:
        googleapiclient.discovery.Resource: Drive service object
        
    Raises:
        SystemExit: If authentication fails
    """
    creds = None

    if os.path.exists(Config.TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(
            Config.TOKEN_PATH, Config.SCOPES
        )

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            print("Refreshing expired token...")
            creds.refresh(Request())
            with open(Config.TOKEN_PATH, 'w', encoding='utf-8') as token:
                token.write(creds.to_json())
            print("Token refreshed successfully")
        else:
            print("Error: No valid credentials found!")
            print(f"Expected token at: {Config.TOKEN_PATH}")
            raise SystemExit(1)

    return build('drive', 'v3', credentials=creds)


print("Configuration and utilities loaded!")

In [ ]:
#@title 3. Download Engine

import threading
from typing import Dict, Callable, Optional

# Thread-safe progress tracking
progress_data: Dict[str, dict] = {}
progress_lock = threading.Lock()


def download_direct(url, save_path, progress_callback=None):
    """Download file directly using requests with retry logic.
    
    Args:
        url: Source URL
        save_path: Destination file path
        progress_callback: Optional callback function(downloaded, total, speed)
        
    Returns:
        str: Path to downloaded file
        
    Raises:
        requests.RequestException: If download fails after retries
    """
    headers = {'User-Agent': Config.USER_AGENT}

    for attempt in range(Config.MAX_RETRIES):
        try:
            response = requests.get(
                url, headers=headers, stream=True,
                allow_redirects=True, timeout=Config.TIMEOUT
            )
            response.raise_for_status()

            filename = get_filename_from_url(url, response)
            if not filename:
                content_type = response.headers.get(
                    'Content-Type', 'application/octet-stream'
                )
                ext = mimetypes.guess_extension(
                    content_type.split(';')[0]
                ) or '.bin'
                filename = f"download_{int(time.time())}{ext}"

            filename = sanitize_filename(filename)
            filepath = os.path.join(save_path, filename)

            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            start_time = time.time()

            with open(filepath, 'wb', encoding=None) as f:
                for chunk in response.iter_content(chunk_size=Config.CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)

                        if progress_callback and total_size > 0:
                            elapsed = time.time() - start_time
                            speed = downloaded / elapsed if elapsed > 0 else 0
                            progress_callback(downloaded, total_size, speed)

            return filepath

        except requests.RequestException as req_err:
            if attempt < Config.MAX_RETRIES - 1:
                print(f"Attempt {attempt+1} failed: {req_err}. Retrying...")
                time.sleep(Config.RETRY_DELAY * (attempt + 1))
            else:
                raise

    raise requests.RequestException("Max retries exceeded")


def download_ytdlp(url, save_path, format_spec='best', progress_callback=None):
    """Download using yt-dlp for video sites.
    
    Args:
        url: Video URL
        save_path: Destination directory
        format_spec: yt-dlp format string
        progress_callback: Optional callback function(downloaded, total, speed)
        
    Returns:
        str: Path to downloaded file
        
    Raises:
        ImportError: If yt-dlp is not installed
        Exception: If download fails
    """
    try:
        import yt_dlp
    except ImportError as exc:
        raise ImportError("yt-dlp not installed") from exc

    outtmpl = os.path.join(save_path, '%(title)s.%(ext)s')

    def progress_hook(d):
        """yt-dlp progress hook."""
        if progress_callback and d['status'] == 'downloading':
            downloaded = d.get('downloaded_bytes', 0)
            total = d.get('total_bytes') or d.get('total_bytes_estimate', 0)
            speed = d.get('speed', 0)
            if total > 0:
                progress_callback(downloaded, total, speed or 0)

    ydl_opts = {
        'format': format_spec,
        'outtmpl': outtmpl,
        'quiet': False,
        'no_warnings': True,
        'retries': Config.MAX_RETRIES,
        'fragment_retries': Config.MAX_RETRIES,
        'progress_hooks': [progress_hook],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        if info:
            if 'requested_downloads' in info:
                return info['requested_downloads'][0]['filepath']
            return ydl.prepare_filename(info)

    raise Exception("yt-dlp extraction failed")


def upload_to_drive(filepath, folder_id=None):
    """Upload file to Google Drive.
    
    Args:
        filepath: Path to local file
        folder_id: Optional destination folder ID
        
    Returns:
        dict: File metadata from Drive API
        
    Raises:
        HttpError: If upload fails
    """
    service = get_drive_service()
    
    filename = os.path.basename(filepath)
    mime_type, _ = mimetypes.guess_type(filepath)
    mime_type = mime_type or 'application/octet-stream'

    file_metadata = {'name': filename}
    if folder_id:
        file_metadata['parents'] = [folder_id]

    media = MediaFileUpload(
        filepath,
        mimetype=mime_type,
        resumable=True,
        chunksize=Config.CHUNK_SIZE
    )

    request = service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id, name, size, webViewLink, mimeType'
    )

    response = None
    while response is None:
        status, response = request.next_chunk()

    return response


print("Download engine loaded!")

In [ ]:
#@title 4. Flask API Server

import json
import uuid
from datetime import datetime
from queue import Queue
from flask import Flask, request, jsonify, Response
from flask_cors import CORS

app = Flask(__name__)
CORS(app, resources={
    r"/api/*": {
        "origins": "*",
        "methods": ["GET", "POST", "DELETE", "OPTIONS"],
        "allow_headers": ["Content-Type", "Accept", "Origin"],
        "expose_headers": ["Content-Type"],
        "supports_credentials": False
    }
})

# Download queue and tracking
download_queue = Queue()
downloads: Dict[str, dict] = {}
downloads_lock = threading.Lock()
server_start_time = datetime.now()


@app.route('/api/health', methods=['GET', 'OPTIONS'])
def health_check():
    """Check server health status."""
    if request.method == 'OPTIONS':
        response = jsonify({'status': 'ok'})
        response.headers.add('Access-Control-Allow-Origin', '*')
        response.headers.add('Access-Control-Allow-Headers', 'Content-Type, Accept')
        response.headers.add('Access-Control-Allow-Methods', 'GET, OPTIONS')
        return response
    uptime = datetime.now() - server_start_time
    return jsonify({
        'status': 'ok',
        'uptime': str(uptime).split('.')[0]
    })


@app.route('/api/download', methods=['POST', 'OPTIONS'])
def start_download():
    """Start a new download."""
    if request.method == 'OPTIONS':
        response = jsonify({'status': 'ok'})
        response.headers.add('Access-Control-Allow-Origin', '*')
        response.headers.add('Access-Control-Allow-Headers', 'Content-Type, Accept')
        response.headers.add('Access-Control-Allow-Methods', 'POST, OPTIONS')
        return response
    data = request.get_json()
    
    if not data or 'url' not in data:
        return jsonify({'error': 'URL required'}), 400
    
    download_id = str(uuid.uuid4())
    
    download_info = {
        'id': download_id,
        'url': data['url'],
        'folder': data.get('folder', Config.DEFAULT_FOLDER),
        'filename': data.get('filename'),
        'format': data.get('format', 'best'),
        'status': 'queued',
        'created_at': datetime.now().isoformat(),
        'progress': {'percent': 0, 'downloaded': 0, 'total': 0, 'speed': 0}
    }
    
    with downloads_lock:
        downloads[download_id] = download_info
    
    download_queue.put(download_id)
    
    return jsonify({'id': download_id, 'status': 'queued'})


@app.route('/api/progress/<download_id>', methods=['GET'])
def get_progress_stream(download_id):
    """SSE endpoint for real-time progress updates."""
    def generate():
        """Generate SSE events."""
        while True:
            with downloads_lock:
                if download_id not in downloads:
                    yield f"data: {{\"error\": \"Download not found\"}}\n\n"
                    break
                
                info = downloads[download_id]
                progress = info.get('progress', {})
                
                event_data = {
                    'status': info['status'],
                    'percent': progress.get('percent', 0),
                    'downloaded': format_size(progress.get('downloaded', 0)),
                    'total': format_size(progress.get('total', 0)),
                    'speed': format_speed(progress.get('speed', 0)),
                    'filename': info.get('result', {}).get('filename', '')
                }
                
                yield f"data: {json.dumps(event_data)}\n\n"
                
                if info['status'] in ['completed', 'failed']:
                    break
            
            time.sleep(0.5)
    
    response = Response(generate(), mimetype='text/event-stream')
    response.headers.add('Access-Control-Allow-Origin', '*')
    response.headers.add('Cache-Control', 'no-cache')
    return response


@app.route('/api/status/<download_id>', methods=['GET'])
def get_download_status(download_id):
    """Get download status."""
    with downloads_lock:
        if download_id not in downloads:
            return jsonify({'error': 'Download not found'}), 404
        return jsonify(downloads[download_id])


@app.route('/api/queue', methods=['GET'])
def get_queue():
    """List all downloads."""
    with downloads_lock:
        return jsonify(list(downloads.values()))


@app.route('/api/download/<download_id>', methods=['DELETE', 'OPTIONS'])
def cancel_download(download_id):
    """Cancel a download."""
    if request.method == 'OPTIONS':
        response = jsonify({'status': 'ok'})
        response.headers.add('Access-Control-Allow-Origin', '*')
        response.headers.add('Access-Control-Allow-Headers', 'Content-Type, Accept')
        response.headers.add('Access-Control-Allow-Methods', 'DELETE, OPTIONS')
        return response
    with downloads_lock:
        if download_id not in downloads:
            return jsonify({'error': 'Download not found'}), 404
        
        downloads[download_id]['status'] = 'cancelled'
        return jsonify({'success': True})


@app.route('/api/folders', methods=['GET', 'POST', 'OPTIONS'])
def handle_folders():
    """List or create Drive folders."""
    if request.method == 'OPTIONS':
        response = jsonify({'status': 'ok'})
        response.headers.add('Access-Control-Allow-Origin', '*')
        response.headers.add('Access-Control-Allow-Headers', 'Content-Type, Accept')
        response.headers.add('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        return response
    
    if request.method == 'GET':
        try:
            service = get_drive_service()
            query = ("mimeType='application/vnd.google-apps.folder' "
                     "and 'root' in parents and trashed=false")
            
            results = service.files().list(
                q=query,
                pageSize=100,
                fields="files(id, name)",
                orderBy="name"
            ).execute()
            
            folders = results.get('files', [])
            return jsonify(folders)
        
        except HttpError as http_err:
            return jsonify({'error': str(http_err)}), 500
    
    # POST - create folder
    data = request.get_json()
    
    if not data or 'name' not in data:
        return jsonify({'error': 'Folder name required'}), 400
    
    try:
        service = get_drive_service()
        
        file_metadata = {
            'name': data['name'],
            'mimeType': 'application/vnd.google-apps.folder'
        }
        
        if data.get('parent'):
            file_metadata['parents'] = [data['parent']]
        
        folder = service.files().create(
            body=file_metadata,
            fields='id, name'
        ).execute()
        
        return jsonify(folder)
    
    except HttpError as http_err:
        return jsonify({'error': str(http_err)}), 500


@app.route('/api/quota', methods=['GET'])
def get_quota():
    """Get Drive storage quota."""
    try:
        service = get_drive_service()
        about = service.about().get(fields="storageQuota").execute()
        
        quota = about.get('storageQuota', {})
        used = int(quota.get('usage', 0))
        total = int(quota.get('limit', 0))
        
        return jsonify({
            'used': format_size(used),
            'total': format_size(total),
            'percent': int((used / total) * 100) if total > 0 else 0
        })
    
    except HttpError as http_err:
        return jsonify({'error': str(http_err)}), 500


def process_download_queue():
    """Background worker to process download queue."""
    while True:
        download_id = download_queue.get()
        
        with downloads_lock:
            if download_id not in downloads:
                continue
            
            info = downloads[download_id]
            
            if info['status'] == 'cancelled':
                continue
            
            info['status'] = 'downloading'
        
        try:
            # Progress callback
            def update_progress(downloaded, total, speed):
                """Update download progress."""
                with downloads_lock:
                    if download_id in downloads:
                        downloads[download_id]['progress'] = {
                            'percent': int((downloaded / total) * 100) if total > 0 else 0,
                            'downloaded': downloaded,
                            'total': total,
                            'speed': speed
                        }
            
            # Determine save path
            folder_name = info.get('folder', Config.DEFAULT_FOLDER)
            save_dir = os.path.join(Config.DRIVE_BASE, folder_name)
            os.makedirs(save_dir, exist_ok=True)
            
            # Download
            url_type = detect_url_type(info['url'])
            
            if url_type == 'video':
                filepath = download_ytdlp(
                    info['url'],
                    save_dir,
                    info.get('format', 'best'),
                    update_progress
                )
            else:
                filepath = download_direct(
                    info['url'],
                    save_dir,
                    update_progress
                )
            
            # Upload to Drive (already in Drive if using Colab mount)
            with downloads_lock:
                info['status'] = 'completed'
                info['completed_at'] = datetime.now().isoformat()
                info['result'] = {
                    'filename': os.path.basename(filepath),
                    'path': filepath,
                    'size': os.path.getsize(filepath)
                }
        
        except Exception as exc:
            with downloads_lock:
                info['status'] = 'failed'
                info['error'] = str(exc)


# Start background worker
worker_thread = threading.Thread(target=process_download_queue, daemon=True)
worker_thread.start()

print("Flask API server configured!")
print("\nAvailable endpoints:")
print("  GET  /api/health")
print("  POST /api/download")
print("  GET  /api/progress/<id>")
print("  GET  /api/status/<id>")
print("  GET  /api/queue")
print("  DELETE /api/download/<id>")
print("  GET  /api/folders")
print("  POST /api/folders")
print("  GET  /api/quota")

In [ ]:
#@title 5. Pinggy.io Tunnel

import subprocess

def start_pinggy_tunnel(port=5000):
    """Start Pinggy tunnel with auto-reconnect.
    
    Args:
        port: Local port to expose
    """
    while True:
        try:
            process = subprocess.Popen(
                ["ssh", "-p", "443", f"-R0:localhost:{port}",
                 "-o", "StrictHostKeyChecking=no",
                 "-o", "ServerAliveInterval=30",
                 "a.pinggy.io"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT
            )
            
            for line in process.stdout:
                line = line.decode().strip()
                if "http" in line.lower():
                    print(f"\n{'='*50}")
                    print(f"TUNNEL URL: {line}")
                    print(f"{'='*50}\n")
                print(line)
            
            process.wait()
        
        except Exception as exc:
            print(f"Tunnel error: {exc}")
        
        print("Reconnecting in 5 seconds...")
        time.sleep(5)


# Run in background thread
tunnel_thread = threading.Thread(target=start_pinggy_tunnel, args=(5000,), daemon=True)
tunnel_thread.start()

print("Pinggy tunnel starting...")
print("Copy the tunnel URL above and use it in your web UI.")
time.sleep(3)  # Give tunnel time to establish

In [ ]:
#@title 6. Start Server

print("="*60)
print("       UNIVERSAL DRIVE UPLOADER - WEB SERVER")
print("="*60)
print("\nServer starting on port 5000...")
print("\nInstructions:")
print("  1. Copy the Pinggy tunnel URL from above")
print("  2. Open your web UI (index.html)")
print("  3. Paste the tunnel URL in settings")
print("  4. Start uploading!")
print("\nPress STOP button to shutdown the server.")
print("="*60)
print()

# Start Flask server
app.run(host='0.0.0.0', port=5000, threaded=True)